# 04 — Reproducible RAG evaluation

This laboratory closes the RAG feedback loop: **run → inspect → promote**. It uses the same `EvaluationApplication` and manifest loader as `raglab-evaluate`; it never invokes the CLI through a subprocess.

The notebook is **hermetic by default**. Only the final opt-in cell can contact PostgreSQL and Ollama, and every artifact it creates is written under `/tmp`.

## Learning outcomes and quick path

By the end you can:

1. load and validate the versioned core manifest;
2. inspect corpus hashes, ground truth, chunk checks, and multi-turn cases;
3. calculate deterministic retrieval metrics on controlled rankings;
4. create compatible baseline and candidate runs through the shared application service;
5. interpret per-case stability, latency deltas, and a conservative verdict; and
6. opt into a real core run, then inspect its JSON and Markdown artifacts.

A candidate is evidence, not an automatically accepted baseline. The operational path is:

1. **Run** a complete candidate.
2. **Inspect** hard failures, identity, per-case metrics, stability, and latency compatibility.
3. **Promote** only a clean, complete, compatible run that you intentionally accept.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

from raglab.evaluation import (
    EvaluationApplication,
    HermeticEvaluationExecutor,
    corpus_fingerprint,
    load_manifest,
)
from raglab.evaluation.metrics import retrieval_metrics


def markdown_table(rows: list[dict[str, Any]], columns: list[str]) -> str:
    def cell(value: object) -> str:
        return str(value).replace("|", "\|").replace("\n", " ")

    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join("---" for _ in columns) + " |"
    body = [
        "| " + " | ".join(cell(row.get(column, "")) for column in columns) + " |"
        for row in rows
    ]
    return "\n".join([header, divider, *body])


def show(value: object) -> None:
    print(json.dumps(value, indent=2, sort_keys=True, default=str))

## 1. Load the shared manifest

The manifest is the evaluation contract. It fixes source identities, source locations, chunk-boundary checks, ground truth, and pipeline configuration. `load_manifest()` performs the same validation used by the CLI. Hashes make the exact corpus observable; PostgreSQL UUIDs are deliberately absent from ground truth.

In [ ]:
manifest = load_manifest(profile="core")
corpus_hash, source_hashes = corpus_fingerprint(manifest)

show(
    {
        "schema_version": manifest.schema_version,
        "profile": manifest.profile,
        "configuration": manifest.config,
        "source_count": len(manifest.sources),
        "case_count": len(manifest.cases),
        "corpus_fingerprint": corpus_hash,
    }
)

source_rows = [
    {
        "source_id": source.id,
        "location": source.location,
        "sha256": source_hashes[source.id][:16] + "…",
    }
    for source in manifest.sources
]
print(markdown_table(source_rows, ["source_id", "location", "sha256"]))

## 2. Inspect ground truth and conversations

Each case names expected evidence with stable `source_id` values and checks answer text through required facts. The abstention case requires neither evidence nor facts. History is part of the retrieval question for the two multi-turn cases; it is not hidden evaluator state.

In [ ]:
case_rows = [
    {
        "case": case.id,
        "expected_sources": ", ".join(case.expected_source_ids) or "—",
        "required_facts": "; ".join(case.required_facts) or "—",
        "abstain": case.should_abstain,
        "history_turns": len(case.history),
    }
    for case in manifest.cases
]
print(
    markdown_table(
        case_rows,
        ["case", "expected_sources", "required_facts", "abstain", "history_turns"],
    )
)

multi_turn_rows = [
    {
        "case": case.id,
        "history": " → ".join(case.history),
        "follow_up": case.query,
    }
    for case in manifest.cases
    if case.history
]
assert len(multi_turn_rows) == 2
print("\nMulti-turn ground truth\n")
print(markdown_table(multi_turn_rows, ["case", "history", "follow_up"]))

## 3. Inspect ingestion controls

`must_separate` anchors describe concepts that should not share a chunk. `must_keep` anchors describe facts that need one coherent chunk. These are semantic chunking checks, not tests of PostgreSQL's physical index structure.

In [ ]:
chunk_rows = [
    {
        "rule": "must separate",
        "check": check.id,
        "source_id": check.source_id,
        "reason": check.reason,
    }
    for check in manifest.must_separate
] + [
    {
        "rule": "must keep",
        "check": check.id,
        "source_id": check.source_id,
        "reason": check.reason,
    }
    for check in manifest.must_keep
]
print(markdown_table(chunk_rows, ["rule", "check", "source_id", "reason"]))

## 4. Calculate deterministic retrieval metrics

The examples below make ranking behavior visible before a complete run. Hit@K asks whether any relevant source appears by K. Recall@5 asks how much expected evidence appears. MRR rewards the first relevant result appearing early. nDCG@5 rewards all relevant results appearing near the top.

An abstention case has no relevant source by design, so a ranking with no required evidence receives full deterministic retrieval credit; generation must independently verify abstention.

In [ ]:
metric_examples = [
    {
        "scenario": "both relevant sources first",
        **retrieval_metrics(
            ["harbor-edge-cache", "meridian-incident-response", "orbit-telemetry"],
            ["harbor-edge-cache", "meridian-incident-response"],
        ),
    },
    {
        "scenario": "one relevant source at rank 2",
        **retrieval_metrics(
            ["orbit-telemetry", "harbor-edge-cache"],
            ["harbor-edge-cache", "meridian-incident-response"],
        ),
    },
    {
        "scenario": "abstention: no expected source",
        **retrieval_metrics([], []),
    },
]
print(
    markdown_table(
        metric_examples,
        [
            "scenario",
            "hit_at_1",
            "hit_at_3",
            "hit_at_5",
            "recall_at_5",
            "mrr",
            "ndcg_at_5",
        ],
    )
)
assert metric_examples[0]["recall_at_5"] == 1.0
assert metric_examples[1]["mrr"] == 0.5

## 5. Build a hermetic baseline and slower candidate

`HermeticEvaluationExecutor` replaces external boundaries with deterministic observations. `EvaluationApplication` still owns the real orchestration: corpus/config fingerprints, ingestion checks, exact and approximate retrieval, three generation repetitions, summaries, and comparison.

Both runs use identical metadata so quality and latency are comparable. Persistence is disabled: this lesson does not create a baseline file or artifacts inside the repository. The candidate changes only its latency scale, so its quality should remain unchanged.

In [ ]:
DEMO_ARTIFACT_DIR = Path("/tmp/raglab-evaluation-notebook-demo")
demo_metadata = {
    "commit": "hermetic-notebook",
    "dirty": False,
    "generation_model": "hermetic-generator",
    "embedding_model": "hermetic-embedding",
    "hardware": {"profile": "deterministic"},
    "hardware_fingerprint": "hermetic-hardware-v1",
}

baseline_application = EvaluationApplication(
    HermeticEvaluationExecutor(),
    artifact_dir=DEMO_ARTIFACT_DIR,
    generation_model="hermetic-generator",
    metadata_provider=lambda: dict(demo_metadata),
)
candidate_application = EvaluationApplication(
    HermeticEvaluationExecutor(latency_scale=1.75),
    artifact_dir=DEMO_ARTIFACT_DIR,
    generation_model="hermetic-generator",
    metadata_provider=lambda: dict(demo_metadata),
)

baseline = baseline_application.run(manifest, persist=False)
candidate = candidate_application.run(manifest, persist=False)
comparison = candidate_application.compare(candidate, baseline)

assert baseline["errors"]["hard"] == []
assert candidate["errors"]["hard"] == []
assert comparison["latency_compatible"] is True
assert comparison["verdict"] == "no_clear_change"
show(
    {
        "baseline": baseline["summary"],
        "candidate": candidate["summary"],
        "comparison": comparison,
        "persisted": False,
    }
)

## 6. Inspect every case before the verdict

The summary is not enough. Per-case inspection reveals which evidence was ranked, whether exact and approximate retrieval agree, whether all three answers satisfy facts/abstention/citations, and where latency changed.

In [ ]:
per_case_rows = []
for baseline_case, candidate_case in zip(
    baseline["cases"], candidate["cases"], strict=True
):
    assert baseline_case["id"] == candidate_case["id"]
    metrics = candidate_case["retrieval"]["metrics"]
    per_case_rows.append(
        {
            "case": candidate_case["id"],
            "Hit@1": metrics["hit_at_1"],
            "Recall@5": metrics["recall_at_5"],
            "MRR": metrics["mrr"],
            "nDCG@5": metrics["ndcg_at_5"],
            "exact agreement": candidate_case["retrieval"]["exact_agreement_at_5"],
            "baseline stability": baseline_case["generation"]["stability"],
            "candidate stability": candidate_case["generation"]["stability"],
            "retrieval ms": candidate_case["retrieval"]["latency_ms"],
        }
    )

print(
    markdown_table(
        per_case_rows,
        [
            "case",
            "Hit@1",
            "Recall@5",
            "MRR",
            "nDCG@5",
            "exact agreement",
            "baseline stability",
            "candidate stability",
            "retrieval ms",
        ],
    )
)

axis_rows = [
    {"axis": axis, **values}
    for axis, values in comparison["axes"].items()
]
print("\nQuality deltas\n")
print(markdown_table(axis_rows, ["axis", "baseline", "candidate", "delta"]))
print(f"\nConservative verdict: **{comparison['verdict']}**")

## 7. Interpret the conservative verdict

`no_clear_change` is correct here: every quality axis is equal, even though the candidate is slower. Latency is reported separately because the hardware fingerprints match. The evaluator never converts quality and speed into one weighted score.

Use the four verdicts as directional summaries:

| Verdict | Meaning |
| --- | --- |
| `improved` | At least one quality axis improved and none regressed. |
| `regressed` | At least one quality axis regressed and none improved. |
| `mixed` | Some quality axes improved while others regressed. |
| `no_clear_change` | The measured quality axes are unchanged. |

A real promotion has a stricter gate than comparison: the run must be complete, full rather than `--reuse-index`, produced from a clean Git worktree, and free of hard failures. Promotion is always explicit; the previous run never becomes the baseline automatically.

## 8. Optional advisory judge

Deterministic facts, abstention, citations, and retrieval evidence remain authoritative. `EvaluationJudge` is an optional protocol for semantic observations that are not yet calibrated as a gate.

The judge model must differ from the generation model. The application saves deterministic answers first, asks the executor to unload the generator, and then invokes the judge sequentially. A judge error or OOM is recorded under advisory errors and does not invalidate the deterministic core. RAGLab does not install or pin a second model for you.

## 9. Optional real core evaluation

Set `RAGLAB_RUN_EVALUATION_NOTEBOOK = "1"` in the next cell, or export `RAGLAB_RUN_EVALUATION_NOTEBOOK=1`, to rebuild the protected `raglab-eval-core` collection through PostgreSQL, Ollama, and BGE.

This cell imports the production executor but still calls `EvaluationApplication` directly. It writes versioned JSON and Markdown only to `/tmp/raglab-evaluation-artifacts/`, then reads those exact artifacts back for inspection. It does not promote a baseline.

In [ ]:
RAGLAB_RUN_EVALUATION_NOTEBOOK = "0"

RUN_REAL_CORE = (
    RAGLAB_RUN_EVALUATION_NOTEBOOK == "1"
    or os.environ.get("RAGLAB_RUN_EVALUATION_NOTEBOOK") == "1"
)
REAL_ARTIFACT_DIR = Path("/tmp/raglab-evaluation-artifacts").resolve()
assert REAL_ARTIFACT_DIR.is_relative_to(Path("/tmp"))

if not RUN_REAL_CORE:
    print(
        "Real core evaluation skipped. "
        "Set RAGLAB_RUN_EVALUATION_NOTEBOOK=1 to enable it."
    )
else:
    from raglab.evaluation.runtime import LiveEvaluationExecutor
    from raglab.retrieval.cli import DEFAULT_DSN

    generation_model = os.environ.get("RAGLAB_GENERATION_MODEL", "qwen3:4b")
    executor = LiveEvaluationExecutor(
        dsn=os.environ.get("RAGLAB_DSN", DEFAULT_DSN),
        generation_model=generation_model,
        embedding_model=os.environ.get(
            "RAGLAB_EMBEDDING_MODEL", "qwen3-embedding:0.6b"
        ),
        ollama_base_url=os.environ.get(
            "RAGLAB_OLLAMA_BASE_URL", "http://127.0.0.1:11434"
        ),
    )
    real_application = EvaluationApplication(
        executor,
        artifact_dir=REAL_ARTIFACT_DIR,
        generation_model=generation_model,
    )
    real_run = real_application.run(load_manifest(profile="core"))
    real_json_path = REAL_ARTIFACT_DIR / f"{real_run['run_id']}.json"
    real_markdown_path = REAL_ARTIFACT_DIR / f"{real_run['run_id']}.md"
    saved_run = json.loads(real_json_path.read_text())

    show(
        {
            "json_artifact": real_json_path,
            "markdown_artifact": real_markdown_path,
            "status": saved_run["status"],
            "hard_errors": saved_run["errors"]["hard"],
            "summary": saved_run["summary"],
        }
    )
    print("\nSaved Markdown summary\n")
    print(real_markdown_path.read_text())

## Interpretation checklist

Before promoting a real candidate, verify:

- [ ] The run is `complete`, not partial, and has no hard failures.
- [ ] The Git worktree was clean, so the recorded commit identifies the evaluated code.
- [ ] Corpus and configuration fingerprints match the baseline before comparing quality.
- [ ] Live source hashes have not changed unexpectedly.
- [ ] Must-separate and must-keep chunk checks all pass.
- [ ] Expected evidence appears by rank 5; inspect multi-evidence cases individually.
- [ ] Exact-versus-HNSW disagreement is understood rather than hidden by the aggregate.
- [ ] Required facts, abstention, and citation checks pass in all three repetitions.
- [ ] Latency is compared only when hardware fingerprints match.
- [ ] A `mixed` verdict is reviewed axis by axis; no weighted score decides for you.
- [ ] Judge observations, if present, remain advisory.
- [ ] Baseline promotion is a deliberate human decision after inspection.

The evaluator supplies evidence. You still own the engineering judgment.